# 01 · Python for Machine Learning

**Module goal.** Get fluent with the four tools you touch every single day in ML work:
NumPy (fast numerical arrays), pandas (tables), the Python built-ins that glue them
together, and clean file handling. We finish with a short section on *prompting an AI
assistant to write good ML code* — a real modern skill.

**How to use this notebook.** Read the short explanation, run the cell below it, then
change a number and run it again. You learn this by poking at it, not by reading.

> Everything here runs on the standard scientific Python stack: `numpy`, `pandas`,
> `matplotlib`, `scikit-learn`.

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)   # reproducibility: same "random" numbers every run
print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 2.4.4 | pandas 3.0.2


## 1. NumPy — the array is the atom of ML

Every dataset you feed a model is ultimately a NumPy array (or something that becomes one):
a matrix `X` of shape `(n_samples, n_features)` and a vector `y` of targets. NumPy is
*vectorized* — you express operations over whole arrays instead of writing Python loops,
which is both shorter and dramatically faster.

In [2]:
# A feature matrix: 5 samples, 3 features
X = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
    [2.0, 1.0, 0.0],
    [3.0, 3.0, 3.0],
])
print("shape:", X.shape, "| dtype:", X.dtype)

# Column-wise statistics (axis=0 collapses rows -> one value per feature)
print("feature means :", X.mean(axis=0))
print("feature stds  :", X.std(axis=0))

# Standardize every feature to mean 0, std 1 — the single most common preprocessing step
X_scaled = (X - X.mean(axis=0)) / X.std(axis=0)
print("scaled means  :", X_scaled.mean(axis=0).round(6))

shape: (5, 3) | dtype: float64
feature means : [3.4 3.8 4.2]
feature stds  : [2.05912603 2.48193473 3.05941171]
scaled means  : [ 0.  0. -0.]


In [3]:
# Boolean indexing: keep rows where feature 0 is above its median
mask = X[:, 0] > np.median(X[:, 0])
print("mask:", mask)
print("kept rows:\n", X[mask])

# Vectorized math beats loops. Compare a manual dot product to np.dot:
w = np.array([0.5, -1.0, 2.0])
manual = sum(X[0, j] * w[j] for j in range(3))
print("manual dot:", manual, "| np.dot:", X[0] @ w)

mask: [False  True  True False False]
kept rows:
 [[4. 5. 6.]
 [7. 8. 9.]]
manual dot: 4.5 | np.dot: 4.5


### Why vectorization matters (a quick timing)

The loop and the vectorized version compute the same thing. On large arrays the vectorized
form can be 50–100x faster because the work happens in optimized C, not the Python interpreter.

In [4]:
import time
big = np.random.rand(1_000_000)

t = time.time()
s = 0.0
for v in big:
    s += v * v
loop_time = time.time() - t

t = time.time()
s2 = np.sum(big * big)
vec_time = time.time() - t

print(f"loop:       {loop_time*1000:7.1f} ms")
print(f"vectorized: {vec_time*1000:7.1f} ms")
print(f"speedup:    {loop_time/vec_time:6.0f}x")

loop:         127.5 ms
vectorized:    37.5 ms
speedup:         3x


## 2. pandas — tables you can actually reason about

NumPy arrays have no column names. pandas adds labels, mixed types, missing-value handling,
grouping, and file I/O. A `DataFrame` is a 2-D labeled table; a `Series` is one column.

Below we build a tiny "sales" table and run the operations you'll reach for constantly:
inspect, filter, derive columns, handle missing values, and aggregate by group.

In [5]:
sales = pd.DataFrame({
    "region":  ["EMEA", "EMEA", "APAC", "APAC", "AMER", "AMER"],
    "revenue": [1200, 950, 1800, np.nan, 700, 1500],
    "cost":    [800, 600, 1100, 900, 500, 1000],
})
sales

,region,revenue,cost
0,EMEA,1200.0,800
1,EMEA,950.0,600
2,APAC,1800.0,1100
3,APAC,NaN,900
4,AMER,700.0,500
5,AMER,1500.0,1000


In [6]:
# Inspect: the three commands you run on every new dataset
print(sales.info())
print("\nSummary stats:\n", sales.describe())

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   region   6 non-null      str    
 1   revenue  5 non-null      float64
 2   cost     6 non-null      int64  
dtypes: float64(1), int64(1), str(1)
memory usage: 276.0 bytes
None

Summary stats:
            revenue         cost
count     5.000000     6.000000
mean   1230.000000   816.666667
std     435.315977   231.660671
min     700.000000   500.000000
25%     950.000000   650.000000
50%    1200.000000   850.000000
75%    1500.000000   975.000000
max    1800.000000  1100.000000


In [7]:
# Handle missing values: count them, then fill revenue with the column mean
print("missing per column:\n", sales.isna().sum(), "\n")
sales["revenue"] = sales["revenue"].fillna(sales["revenue"].mean())

# Derive a new column
sales["profit"] = sales["revenue"] - sales["cost"]

# Filter with a readable query
big_deals = sales.query("revenue >= 1000")
print("rows with revenue >= 1000:", len(big_deals))
sales

missing per column:
 region     0
revenue    1
cost       0
dtype: int64 

rows with revenue >= 1000: 4


,region,revenue,cost,profit
0,EMEA,1200.0,800,400.0
1,EMEA,950.0,600,350.0
2,APAC,1800.0,1100,700.0
3,APAC,1230.0,900,330.0
4,AMER,700.0,500,200.0
5,AMER,1500.0,1000,500.0


In [8]:
# Group and aggregate: total & average profit per region
summary = (
    sales.groupby("region", as_index=False)
         .agg(total_profit=("profit", "sum"),
              avg_revenue=("revenue", "mean"))
         .sort_values("total_profit", ascending=False)
)
summary

,region,total_profit,avg_revenue
1,APAC,1030.0,1515.0
2,EMEA,750.0,1075.0
0,AMER,700.0,1100.0


## 3. Python built-ins that carry ML code

Comprehensions, dictionaries, and f-strings show up everywhere: mapping model names to
scores, building grids of hyperparameters, formatting metrics for a report.

In [9]:
# A dict is the natural home for "name -> score" results
model_accuracy = {"baseline": 0.78, "logreg": 0.85, "random_forest": 0.91}

# .get() with a default avoids KeyError on a missing model
print("logreg:", model_accuracy.get("logreg"))
print("svm:   ", model_accuracy.get("svm", "not trained yet"))

# Sort models best-first and print a clean report with f-strings
for name, acc in sorted(model_accuracy.items(), key=lambda kv: kv[1], reverse=True):
    print(f"{name:<15} {acc:.1%}")

logreg: 0.85
svm:    not trained yet
random_forest   91.0%
logreg          85.0%
baseline        78.0%


In [10]:
# List comprehension: square only the even numbers 1..10
evens_squared = [n**2 for n in range(1, 11) if n % 2 == 0]
print(evens_squared)

# Dict comprehension: build a hyperparameter label -> value map
grid = {f"C={c}": c for c in [0.01, 0.1, 1, 10]}
print(grid)

[4, 16, 36, 64, 100]
{'C=0.01': 0.01, 'C=0.1': 0.1, 'C=1': 1, 'C=10': 10}


## 4. Files with `pathlib`

`pathlib.Path` builds file paths that work on any OS (no hand-glued `"/"`), and reads/writes
text without ceremony. Use it instead of raw string paths.

In [11]:
from pathlib import Path

work = Path("/home/claude/scratch")
work.mkdir(parents=True, exist_ok=True)     # create the folder if needed

data_path = work / "notes.txt"              # OS-safe join with "/"
data_path.write_text("experiment 1: roc_auc=0.92\n", encoding="utf-8")

print("contents:", data_path.read_text(encoding="utf-8").strip())

# Save the sales table to CSV and read it back — the everyday I/O round-trip
csv_path = work / "sales.csv"
sales.to_csv(csv_path, index=False)
reloaded = pd.read_csv(csv_path)
print("reloaded rows:", len(reloaded))

contents: experiment 1: roc_auc=0.92
reloaded rows: 6


## 5. Prompting an AI assistant for good ML code

You will often ask an assistant to write the boilerplate. Vague prompts get vague code.
Five habits turn a weak prompt into a strong one:

1. **State the task concretely** — name the variable and the output.
2. **Name the library** — "use pandas", "use scikit-learn".
3. **Define inputs and outputs** — shapes, dtypes, the return value.
4. **Add constraints** — "no explicit loops", "handle missing values".
5. **Ask for runnable code** — including imports.

| Weak | Strong |
|---|---|
| "clean the data" | "In `sales_df`, fill missing `revenue` with the column mean and return the DataFrame." |
| "train a model" | "Train a scikit-learn `LogisticRegression` on `X_train, y_train`, print test accuracy." |
| "make a plot" | "Plot a histogram of `sales_df['profit']` with 20 bins, labeled axes, using matplotlib." |

The reason it works: a model (or a colleague) can only be as precise as the request.
Specificity is the whole game.

## Recap

- **NumPy**: vectorized arrays; `axis=0` = per-feature; standardize with `(X - mean)/std`.
- **pandas**: labeled tables; `info`/`describe`/`isna`, `query`, `groupby().agg()`.
- **Built-ins**: dicts for results, comprehensions for grids, f-strings for reports.
- **pathlib**: OS-safe paths and painless read/write.
- **Prompting**: task + library + I/O + constraints + "runnable".

**Next:** `02 — Math Foundations`, where these arrays become gradient descent and probability.